# Documentacion de lo realizado
**Objetivo**: Obtener un conjunto de datos limpio, geolocalizado y representativo del tráfico de la M-30 para el entrenamiento de modelos predictivos de series temporales.

1. Pasos Realizados (Fases 1 al 4)
- Carga y Fusión de Datos: Se han importado los metadatos de los sensores (ubicaciones_m30.csv) y las series temporales históricas (dataset_completo.parquet). Ambos conjuntos se cruzaron mediante un inner join utilizando el identificador único del sensor (id).
- Filtrado Temporal: Se descartaron los datos anteriores al 1 de enero de 2024.
- Control de Calidad Proactivo: Antes de seleccionar los sensores, se auditó el dataset completo frente a un calendario continuo. Se descartaron sensores con completitud menor al 95%, vacíos temporales mayores a 72 horas continuas (como el caso anómalo del sensor 1018) y aquellos con más de un 5% de ceros. Esto garantizó una base de 176 sensores completamente sanos.
- Ingeniería de Características: Se calculó la media y, fundamentalmente, la varianza de la intensidad para cada sensor validado.
- Selección Espacial: Se identificaron los 4 sensores definitivos y validados que conformarán el objeto de estudio.

2. Justificación Metodológica de las Decisiones
- Uso de datos pre-agregados a nivel horario: Trabajar con frecuencias de 1 hora (en lugar de los 15 minutos originales) actúa como un filtro natural que elimina el ruido de alta frecuencia (pequeños atascos, semáforos, incidentes efímeros). Esto optimiza el coste computacional y permite a los modelos (estadísticos y Deep Learning) centrarse en capturar los patrones macro-estacionales (horas punta diurnas, valles nocturnos).
- Acotación Temporal (Desde 2024): Para garantizar que los algoritmos predictivos aprendan de la dinámica vial actual, se ha restringido el historial al año 2024 en adelante, evitando que patrones obsoletos introduzcan sesgos en los pesos de las redes neuronales o en los coeficientes autorregresivos.
- Filtrado de Calidad por "Rechazo Temprano" (Early Rejection): Aplicar los controles de calidad de forma proactiva (antes de la selección) asegura que los modelos aprendan exclusivamente de patrones reales. Realizar imputaciones masivas sobre sensores rotos habría destruido la varianza real de la serie y causado el fracaso del entrenamiento.
- Metodología de Selección de Sensores (Heurística Espacial + Varianza): Para cumplir con el objetivo de obtener 4 nodos representativos (Norte, Sur, Este y Oeste), se descartó el uso de algoritmos de clustering no supervisado (como K-Means), ya que estos agrupan por pura similitud matemática y no garantizan una distribución geográfica equilibrada. En su lugar, se implementó un algoritmo heurístico: 
- Se utilizaron los percentiles 20 y 80 de las coordenadas (latitud y longitud) para aislar los cuadrantes geográficos extremos de la M-30.
- Dentro de cada cuadrante, se seleccionó el nodo con la mayor varianza de intensidad histórica. Esto asegura matemáticamente que los modelos se entrenarán con series temporales ricas en fluctuaciones, proporcionando una señal fuerte y clara para poner a prueba la capacidad predictiva de los modelos ARIMA y LSTM.


# Fase 1: Cargue y visualizacion inicial de los datos

In [5]:
import pandas as pd 

# Cargamos el fichero con las ubicaciones de los sensores M30
df_ubicaciones = pd.read_csv('../data/extra/ubicaciones_m30.csv')

# Cargamos fichero parquet con los datos historicos con los sensosres M30
df_trafico = pd.read_parquet('../data/processed/dataset_completo.parquet', engine='pyarrow')

# 3. Inspección inicial: Identificamos las dimensiones de nuestros datos
print("--- DIMENSIONES DE LOS DATOS ---")
print(f"Catálogo de Sensores (Ubicaciones): {df_ubicaciones.shape[0]} filas y {df_ubicaciones.shape[1]} columnas.")
print(f"Registros Históricos (Tráfico): {df_trafico.shape[0]:,} filas y {df_trafico.shape[1]} columnas.\n".replace(',', '.'))

# 4. Ver las primeras líneas de cada DataFrame
print("--- MUESTRA DE UBICACIONES ---")
# 'display()' es una función especial de Jupyter que formatea la tabla de forma visual
display(df_ubicaciones.head(3)) 

print("\n--- MUESTRA DE TRÁFICO ---")
display(df_trafico.head(3))

--- DIMENSIONES DE LOS DATOS ---
Catálogo de Sensores (Ubicaciones): 353 filas y 11 columnas.
Registros Históricos (Tráfico): 9.624.596 filas y 6 columnas.

--- MUESTRA DE UBICACIONES ---


,tipo_elem,distrito,id,cod_cent,nombre,utm_x,utm_y,longitud,latitud,tipo_elem_aux,tipo_elem_m30
0,M30,3.0,6668,PM10768,PM10768,"444001,8459","4474331,118",-3.660063,40.417722,M30,1
1,M30,3.0,6669,PM10766,PM10766,"443977,812","4474303,082",-3.660344,40.417468,M30,1
2,M30,2.0,6688,PM11303,PM11303,"441272,9676","4470633,619",-3.691886,40.384225,M30,1



--- MUESTRA DE TRÁFICO ---


,id,fecha,intensidad,ocupacion,carga,vmed
0,1012,2022-01-01 00:00:00,36.00,1.00,0.0,11.00
1,1012,2022-01-01 01:00:00,509.25,4.25,0.0,53.25
2,1012,2022-01-01 02:00:00,441.00,3.75,0.0,51.00


# Fase 2: Preparacion de los datos

In [6]:
import numpy as np

print("Iniciando Fase 2: Data Preparation Avanzada con Filtro de Calidad...")

# 1. Asegurar formato de tiempo correcto en la serie temporal
df_trafico['fecha'] = pd.to_datetime(df_trafico['fecha'])

# 2. Filtrado Temporal Metodologico
fecha_inicio = pd.to_datetime('2024-01-01 00:00:00')
df_trafico_2024 = df_trafico[df_trafico['fecha'] >= fecha_inicio].copy()

# 3. Hacemos el Join con los datos espaciales usando el ID
df_maestro_inicial = pd.merge(
    left=df_trafico_2024, 
    right=df_ubicaciones[['id', 'latitud', 'longitud', 'nombre']], 
    on='id', 
    how='inner'
)

# 4. Evaluacion de completitud de los sensores UPSTREAM
# Ordenamos cronológicamente por sensor y fecha para medir saltos de tiempo auténticos
df_maestro_inicial = df_maestro_inicial.sort_values(by=['id', 'fecha'])

# Calculamos la diferencia en horas entre filas consecutivas de un mismo id
# Con diff() calculamos la cantidad de horas entre una fecha y la anterior
df_maestro_inicial['diff_horas'] = df_maestro_inicial.groupby('id')['fecha'].diff().dt.total_seconds() / 3600

# Hacemos calculo del numero de huecos
df_maestro_inicial['hueco_horas'] = df_maestro_inicial['diff_horas'] - 1 

# Definimos las horas teoricas perfectas que deberian existir desde 2024 hasta el final
fecha_fin = df_maestro_inicial['fecha'].max()
total_esperado = len(pd.date_range(start=fecha_inicio, end=fecha_fin, freq='1h'))

# Agrupamos todo el universo de sensores para auditar su salud matematica
# Luego usamos el .agg para poder operar sobre cada columna
qa_sensores = df_maestro_inicial.groupby('id').agg(
    total_real=('intensidad', 'count'),
    ceros_totales=('intensidad', lambda x: (x == 0).sum()),
    max_gap_horas=('hueco_horas', 'max')
).reset_index()

# Convertimos los valores absolutos en indicadores de calidad relativos (%)
qa_sensores['completitud_pct'] = (qa_sensores['total_real'] / total_esperado) * 100
qa_sensores['ceros_pct'] = (qa_sensores['ceros_totales'] / qa_sensores['total_real']) * 100

# 5. APLICACION DE REGLAS DE NEGOCIO (Los filtros minimos admisibles)
UMBRAL_COMPLETITUD = 95.0  # Mínimo 95% de datos presentes
UMBRAL_GAP = 72.0          # No se permiten vacíos continuos mayores a 3 días (72 horas)
UMBRAL_CEROS = 5.0         # No se permiten fallos intermitentes (marcar cero) mayores al 5% del tiempo

# Filtramos la lista de IDs que superan el control de calidad
sensores_aptos = qa_sensores[
    (qa_sensores['completitud_pct'] >= UMBRAL_COMPLETITUD) & 
    (qa_sensores['max_gap_horas'] <= UMBRAL_GAP) & 
    (qa_sensores['ceros_pct'] <= UMBRAL_CEROS)
]['id'].tolist()

# 6. CREACION DEL DATAFRAME MAESTRO DEFINITIVO
# Nos quedamos unicamente con los datos de los sensores saludables
df_maestro = df_maestro_inicial[df_maestro_inicial['id'].isin(sensores_aptos)].copy()

# Limpiamos columnas auxiliares de calculo para optimizar memoria RAM
df_maestro = df_maestro.drop(columns=['diff_horas', 'hueco_horas'])

print("\n--- REPORTE DE CONTROL DE CALIDAD PROACTIVO ---")
print(f"-> Total de sensores analizados en la M-30: {qa_sensores.shape[0]}")
print(f"-> Sensores que superan los criterios de calidad (APTOS): {len(sensores_aptos)}")
print(f"-> Sensores eliminados del flujo por anomalías (RECHAZADOS): {qa_sensores.shape[0] - len(sensores_aptos)}")
print(f"-> Tamaño del DataFrame Maestro finalizado: {df_maestro.shape[0]:,} filas.".replace(',', '.'))

# Verificación de que el sensor problemático 1018 ha sido purgado con éxito
if 1018 in sensores_aptos:
    print("\n⚠️ ALERTA: El sensor 1018 sigue en la lista. Revisa los filtros.")
else:
    print("\n✅ ÉXITO METODOLÓGICO: El sensor anomalía (ID 1018) ha sido vetado proactivamente de la base de datos.")

Iniciando Fase 2: Data Preparation Avanzada con Filtro de Calidad...

--- REPORTE DE CONTROL DE CALIDAD PROACTIVO ---
-> Total de sensores analizados en la M-30: 344
-> Sensores que superan los criterios de calidad (APTOS): 176
-> Sensores eliminados del flujo por anomalías (RECHAZADOS): 168
-> Tamaño del DataFrame Maestro finalizado: 3.299.744 filas.

✅ ÉXITO METODOLÓGICO: El sensor anomalía (ID 1018) ha sido vetado proactivamente de la base de datos.


# Fase 3: Selección Espacial de 4 sensores representativos (Norte, Sur, Este y Oeste).

In [7]:
print("Iniciando Fase 3: Selección de Nodos Estratégicos sobre sensores APTOS...")

# 1. Agrupamos por sensor (id) para calcular la varianza y mantener su ubicacion
df_kpis = df_maestro.groupby('id').agg(
    intensidad_media=('intensidad', 'mean'),
    intensidad_varianza=('intensidad', 'var'),
    latitud=('latitud', 'first'),
    longitud=('longitud', 'first'),
    nombre=('nombre', 'first')
).reset_index()

# 2. Definimos los umbrales geográficos (percentiles 20 y 80) para aislar los extremos
# Evaluar si queremos puntos que sean mas estrictamente norte y estrictmente sur 
lat_80 = df_kpis['latitud'].quantile(0.80) # Límite Norte
lat_20 = df_kpis['latitud'].quantile(0.20) # Límite Sur
lon_80 = df_kpis['longitud'].quantile(0.80) # Límite Este
lon_20 = df_kpis['longitud'].quantile(0.20) # Límite Oeste

# 3. Creamos subconjuntos para cada zona cardinal
sensores_norte = df_kpis[df_kpis['latitud'] >= lat_80]
sensores_sur   = df_kpis[df_kpis['latitud'] <= lat_20]
sensores_este  = df_kpis[df_kpis['longitud'] >= lon_80]
sensores_oeste = df_kpis[df_kpis['longitud'] <= lon_20]

# 4. Seleccionamos el sensor con mayor varianza de trafico en cada zona
seleccion = []

# Creamos funcion para obtener el sensor que mejor se adapte a la condiciones
def obtener_mejor_sensor(df_zona, etiqueta):
    # Obtenemos el indice del sensor con la mayor varianza
    idx_max_var = df_zona['intensidad_varianza'].idxmax()
    mejor_sensor = df_zona.loc[idx_max_var].copy()
    mejor_sensor['zona_cardinal'] = etiqueta
    return mejor_sensor

# Hacemos un append del mejor sensor id de acuerdo con las condiciones
seleccion.append(obtener_mejor_sensor(sensores_norte, 'Norte'))
seleccion.append(obtener_mejor_sensor(sensores_sur, 'Sur'))
seleccion.append(obtener_mejor_sensor(sensores_este, 'Este'))
seleccion.append(obtener_mejor_sensor(sensores_oeste, 'Oeste'))

# 5. Consolidamos el DataFrame final con los 4 sensores estratégicos
df_4_sensores = pd.DataFrame(seleccion)
# Reordenamos columnas para que sea más legible
df_4_sensores = df_4_sensores[['zona_cardinal', 'id', 'nombre', 'intensidad_media', 'intensidad_varianza', 'latitud', 'longitud']]

print("\n--- NUEVA SELECCIÓN FINAL: 4 SENSORES ESTRATÉGICOS (100% VALIDADOS) ---")
display(df_4_sensores)

Iniciando Fase 3: Selección de Nodos Estratégicos sobre sensores APTOS...

--- NUEVA SELECCIÓN FINAL: 4 SENSORES ESTRATÉGICOS (100% VALIDADOS) ---


,zona_cardinal,id,nombre,intensidad_media,intensidad_varianza,latitud,longitud
38,Norte,6642,PM10091,3001.810781,4.093660e+06,40.476038,-3.674395
54,Sur,6676,PM10981,3439.004640,3.321834e+06,40.400672,-3.667784
27,Este,3820,PM10561,3087.958472,3.042887e+06,40.435670,-3.659971
119,Oeste,6782,PM32351,2673.931043,3.001860e+06,40.455445,-3.744978


# Fase 4: Visualizacion en el mapa de Madrid

In [8]:
import plotly.express as px

print("Generando mapa interactivo con estilo Carto-Positron en el navegador...")

# Usamos px.scatter_map c
fig = px.scatter_map(
    df_4_sensores, 
    lat="latitud", 
    lon="longitud", 
    hover_name="zona_cardinal", 
    # Datos que aparecerán al pasar el cursor
    hover_data={
        "id": True, 
        "nombre": True, 
        "intensidad_media": ":.2f",
        "intensidad_varianza": ":.2e",
        "latitud": False, 
        "longitud": False
    },
    color="zona_cardinal",
    # Mapeo de colores: asignamos colores claros y distintivos
    color_discrete_map={
        "Norte": "#FF0000",   # Rojo
        "Sur": "#FF8C00",     # Naranja
        "Este": "#0000FF",    # Azul
        "Oeste": "#008000"    # Verde
    },
    zoom=11, 
    map_style="carto-positron", # Estilo limpio y profesional solicitado
    title="Sensores Estratégicos Seleccionados para el TFM (M30)"
)

# Ajuste del tamaño de los puntos y opacidad para mejorar la visibilidad
fig.update_traces(marker=dict(size=12, opacity=0.8))

# Ajuste de márgenes
fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})

# Abrir en el navegador para asegurar la visualización
fig.show(renderer="browser")

Generando mapa interactivo con estilo Carto-Positron en el navegador...
